In [ ]:
import pandas as pd
df = pd.read_csv("buys_computer.csv")

df

In [ ]:
x = df.drop(["RID","class:buys_computer"], axis=1)
y = df["class:buys_computer"]
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2)
target = "class:buys_computer"

attributes = list(x_train.columns)

In [ ]:
from collections import Counter
import math

def entropy(y):
    counts = Counter(y)
    total = len(y)
    ent = 0.0

    for count in counts.values():
        p = count / total
        ent -= p * math.log2(p)

    return ent

##information gain

In [ ]:
def information_gain(data,attribute,target):
    total_entropy = entropy(data[target])
    values = data[attribute].unique()
    weighted_entropy = 0
    for value in values:
        subset = data[data[attribute] == value]

        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy

In [ ]:
def build_id3(data, attributes, target):
    classes = data[target]

    # If all examples belong to the same class
    if len(classes.unique()) == 1:
        return classes.iloc[0]

    # If no attributes are left, return the majority class
    if len(attributes) == 0:
        return classes.mode()[0]

    gains = {}

    for attr in attributes:
        gains[attr] = information_gain(
            data,
            attr,
            target
        )

    best_attr = max(gains, key=gains.get)

    tree = {best_attr: {}}

    remaining_attrs = [
        attr for attr in attributes
        if attr != best_attr
    ]

    for value in data[best_attr].unique():
        subset = data[data[best_attr] == value]

        if len(subset) == 0:
            tree[best_attr][value] = classes.mode()[0]
        else:
            tree[best_attr][value] = build_id3(
                subset,
                remaining_attrs,
                target
            )

    return tree

##c4.5

In [ ]:
import math

def gain_ratio(data, attribute, target):
    gain = information_gain(
        data,
        attribute,
        target
    )

    split_info = 0

    for value in data[attribute].unique():
        p = len(
            data[data[attribute] == value]
        ) / len(data)

        split_info -= p * math.log2(p)

    if split_info == 0:
        return 0

    return gain / split_info

In [ ]:
def build_c45(data, attributes, target):
    classes = data[target]

    # If all samples belong to one class
    if len(classes.unique()) == 1:
        return classes.iloc[0]

    # If no attributes remain, return majority class
    if len(attributes) == 0:
        return classes.mode()[0]

    ratios = {}

    for attr in attributes:
        ratios[attr] = gain_ratio(
            data,
            attr,
            target
        )

    best_attr = max(ratios, key=ratios.get)

    tree = {best_attr: {}}

    remaining_attrs = [
        a for a in attributes
        if a != best_attr
    ]

    for value in data[best_attr].unique():
        subset = data[data[best_attr] == value]

        if len(subset) == 0:
            tree[best_attr][value] = classes.mode()[0]
        else:
            tree[best_attr][value] = build_c45(
                subset,
                remaining_attrs,
                target
            )

    return tree

##CART

In [ ]:
def gini(y):

    counts = Counter(y)

    total = len(y)

    impurity = 1

    for count in counts.values():

        p = count / total

        impurity -= p ** 2

    return impurity

In [ ]:
def best_cart_split(data, attributes, target):

    best_attr = None

    best_score = float("inf")

    for attr in attributes:

        score = 0

        for value in data[attr].unique():

            subset = data[
                data[attr] == value
            ]

            score += (
                len(subset) / len(data)
            ) * gini(subset[target])

        if score < best_score:

            best_score = score

            best_attr = attr

    return best_attr

In [ ]:
def build_cart(data, attributes, target):

    classes = data[target]

    if len(classes.unique()) == 1:
        return classes.iloc[0]

    if len(attributes) == 0:
        return classes.mode()[0]

    best_attr = best_cart_split(
        data,
        attributes,
        target
    )

    tree = {best_attr: {}}

    remaining = [
        a for a in attributes
        if a != best_attr
    ]

    for value in data[best_attr].unique():

        subset = data[
            data[best_attr] == value
        ]

        tree[best_attr][value] = build_cart(
            subset,
            remaining,
            target
        )

    return tree

In [ ]:
def predict(tree, instance):
    if not isinstance(tree, dict):
        return tree

    root = next(iter(tree))

    value = instance[root]

    if value not in tree[root]:
        return None

    return predict(
        tree[root][value],
        instance
    )

In [ ]:
def accuracy(actual, pred):

    correct = sum(
        a == p
        for a, p in zip(actual, pred)
    )

    return correct / len(actual)

In [ ]:
def precision_recall_f1(
        actual,
        pred,
        positive="Yes"):

    TP = FP = FN = 0

    for a, p in zip(actual, pred):

        if p == positive and a == positive:
            TP += 1

        elif p == positive and a != positive:
            FP += 1

        elif p != positive and a == positive:
            FN += 1

    precision = TP/(TP+FP) if TP+FP else 0

    recall = TP/(TP+FN) if TP+FN else 0

    f1 = (
        2*precision*recall/
        (precision+recall)
        if precision+recall else 0
    )

    return precision, recall, f1

In [ ]:
def tree_depth(tree):

    if not isinstance(tree, dict):
        return 1

    root = next(iter(tree))

    return 1 + max(
        tree_depth(subtree)
        for subtree in tree[root].values()
    )

In [ ]:
def leaf_count(tree):

    if not isinstance(tree, dict):
        return 1

    root = next(iter(tree))

    return sum(
        leaf_count(subtree)
        for subtree in tree[root].values()
    )

In [ ]:
target = "class:buys_computer"

attributes = [
    col for col in df.columns
    if col not in ["RID", target]
]

In [ ]:
import numpy as np
import math
import time
from collections import Counter

train_data = x_train.copy()
train_data[target] = y_train

start = time.time()

id3_tree = build_id3(
    train_data,
    attributes,
    target
)

id3_time = time.time() - start

In [ ]:
start = time.time()

c45_tree = build_c45(
    train_df,
    attributes,
    target
)

c45_time = time.time() - start

In [ ]:
for name, tree in [
    ("ID3", id3_tree),
    ("C4.5", c45_tree),
    ("CART", cart_tree)
]:

    preds = []

    for _, row in x_test.iterrows():
        preds.append(predict(tree, row))

    acc = accuracy(y_test, preds)

    p, r, f1 = precision_recall_f1(
        y_test,
        preds
    )

    depth = tree_depth(tree)
    leaves = leaf_count(tree)

    print("\n", name)
    print("Accuracy :", round(acc, 4))
    print("Precision :", round(p, 4))
    print("Recall :", round(r, 4))
    print("F1 :", round(f1, 4))
    print("Depth :", depth)
    print("Leaves :", leaves)

In [ ]:
print(id3_tree)
print(c45_tree)
print(cart_tree)